## Developing an Automatic Text Summarizer (Extractive + Abstractive Comparison)


In [2]:
# importing all the required libraries
# Core
import re
import numpy as np
import pandas as pd

# NLP
import nltk
import spacy

# NLTK
from nltk.tokenize import sent_tokenize

# Sklearn (later use)
from sklearn.feature_extraction.text import TfidfVectorizer

# Transformers (later use)
from transformers import pipeline

# Evaluation (bonus)
from rouge_score import rouge_scorer

# Downloading the required NLTK resources
nltk.download("punkt")

# Loading the Spacy model
nlp = spacy.load("en_core_web_sm")

[nltk_data] Downloading package punkt to C:\Users\Saif
[nltk_data]     Ullah\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [3]:
# load the dataset
df = pd.read_csv("bbc_news_dataset.csv")

In [4]:
#checking the dataset
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2225 entries, 0 to 2224
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   text    2225 non-null   object
dtypes: object(1)
memory usage: 17.5+ KB


In [5]:
#checking the first few rows of the dataset
df.head()

,text
0,"Wal-Mart, the largest US retailer, has said it..."
1,Brute force budget cuts or spending caps would...
2,Trouble-hit Mitsubishi Motors is in talks with...
3,Typically it takes about three years from when...
4,"More than 2,000 business and political leaders..."


In [ ]:
# preprocessing the text data
def preprocess_text(text):
    """
    Clean preprocessing:
    - Lowercase
    - Keep numbers (important for summaries)
    - Remove noise
    - Sentence tokenization
    - Lemmatization
    - Remove stopwords & punctuation
    """
    
    # Lowercase
    text = text.lower()
    
    # Keep numbers + letters + dot
    text = re.sub(r'[^a-zA-Z0-9. ]', '', text)
    
    # Sentence tokenization
    sentences = sent_tokenize(text)
    
    cleaned_sentences = []
    
    # Lemmatization + remove stopwords & punctuation
    for sent in sentences:
        doc = nlp(sent)
        
        words = [
            token.lemma_
            for token in doc
            if not token.is_stop
            and not token.is_punct
        ]
        
        cleaned_sentences.append(" ".join(words))
    
    cleaned_text = " ".join(cleaned_sentences)
    
    return cleaned_text, sentences

In [7]:
#apply the preprocessing to the dataset

# Store only cleaned text
df["cleaned_text"] = df["text"].apply(lambda x: preprocess_text(x)[0])

# Store sentences separately (optional)
df["sentences"] = df["text"].apply(lambda x: preprocess_text(x)[1])

In [8]:
# show the maximum columns and rows
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

In [9]:
# checking the data after preprocessing
df.head()

,text,cleaned_text,sentences
0,"Wal-Mart, the largest US retailer, has said it...",walmart large retailer say december sale expec...,[walmart the largest us retailer has said its ...
1,Brute force budget cuts or spending caps would...,brute force budget cut spending cap illserve n...,[brute force budget cuts or spending caps woul...
2,Trouble-hit Mitsubishi Motors is in talks with...,troublehit mitsubishi motors talk french carma...,[troublehit mitsubishi motors is in talks with...
3,Typically it takes about three years from when...,typically take year decision take new model hi...,[typically it takes about three years from whe...
4,"More than 2,000 business and political leaders...",2000 business political leader globe arrive sw...,[more than 2000 business and political leaders...


In [10]:
def extractive_summary(text, num_sentences=2):
    """
    TF-IDF based extractive summarization (IMPROVED)
    """
    
    # Step 1: Get original sentences
    _, original_sentences = preprocess_text(text)
    
    # Step 2: Clean each sentence individually
    cleaned_sentences = [preprocess_text(sent)[0] for sent in original_sentences]
    
    # Step 3: TF-IDF on cleaned sentences
    vectorizer = TfidfVectorizer()
    tfidf_matrix = vectorizer.fit_transform(cleaned_sentences)
    
    # Step 4: Score sentences
    scores = np.array(tfidf_matrix.sum(axis=1)).flatten()
    
    # Step 5: Rank sentences
    ranked = np.argsort(scores)[::-1]
    
    selected_idx = sorted(ranked[:num_sentences])
    
    # Step 6: Use ORIGINAL sentences for output
    summary = " ".join([original_sentences[i] for i in selected_idx])
    
    return summary, selected_idx

In [11]:
# apply the extractive summarization to the dataset
df["extractive_summary"] = df["text"].apply(lambda x: extractive_summary(x)[0])

In [12]:
# Take one sample
text = df["text"].iloc[0]

In [ ]:
# Load the T5 model for summarization (important to load once)
summarizer = pipeline("summarization", model="t5-small")

#create the abstractive summary function
def abstractive_summary(text, max_len=80, min_len=30):
    
    # Limit input size (important for T5)
    text = text[:1000]
    
    summary = summarizer(
        text,
        max_length=max_len,
        min_length=min_len,
        do_sample=False
    )
    
    return summary[0]['summary_text']

'(MaxRetryError("HTTPSConnectionPool(host='huggingface.co', port=443): Max retries exceeded with url: /t5-small/resolve/main/config.json (Caused by ConnectTimeoutError(<HTTPSConnection(host='huggingface.co', port=443) at 0x2c500248380>, 'Connection to huggingface.co timed out. (connect timeout=10)'))"), '(Request ID: 7a49a3c6-1c4a-4e3c-bb92-7e0d9171527b)')' thrown while requesting HEAD https://huggingface.co/t5-small/resolve/main/config.json
Retrying in 1s [Retry 1/5].
c:\MiniForge\envs\intern_env\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Saif Ullah\.cache\huggingface\hub\models--t5-small. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://h

In [ ]:
# rouge_score evaluation function, compute ROUGE scores between reference and generated summaries.
def compute_rouge(reference, generated):
    """
    Evaluate summary quality using ROUGE metrics.

    ROUGE measures similarity between:
    - reference summary (ground truth)
    - generated summary (model output)

    Metrics used:
    - ROUGE-1: unigram overlap
    - ROUGE-2: bigram overlap
    - ROUGE-L: longest common subsequence

    Returns:
    - dictionary of ROUGE scores
    """
    
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    
    scores = scorer.score(reference, generated)
    
    return scores

In [ ]:
def highlight_sentences(text, selected_idx):
    """
    Highlight important sentences in the original text based on extractive summarization.

    Purpose:
    - Visually distinguish sentences that were selected by the extractive model
    - Helps in comparing full text vs summary selection
    - Useful for analysis and debugging of TF-IDF ranking

    Parameters:
    - text (str): Original input text
    - selected_idx (list): Indices of sentences selected by extractive summarizer

    Returns:
    - str: Full text where selected sentences are marked as "[SELECTED]"
    """

    # Split the full text into individual sentences
    sentences = sent_tokenize(text)

    # List to store processed sentences (highlighted or normal)
    highlighted = []

    # Loop through each sentence with its index
    for i, sent in enumerate(sentences):

        # If sentence index is in selected list, mark it as important
        if i in selected_idx:
            highlighted.append(f"[SELECTED] {sent}")

        # Otherwise keep it unchanged
        else:
            highlighted.append(sent)

    # Join all sentences back into a single formatted text block
    return "\n".join(highlighted)

In [16]:
# Extractive
ext_summary, selected_idx = extractive_summary(text, num_sentences=2)

# Abstractive
abs_summary = abstractive_summary(text)

# Highlight
highlighted_text = highlight_sentences(text, selected_idx)

print("========== ORIGINAL TEXT ==========\n")
print(text)

print("\n========== EXTRACTIVE SUMMARY ==========\n")
print(ext_summary)

print("\n========== ABSTRACTIVE SUMMARY ==========\n")
print(abs_summary)

print("\n========== HIGHLIGHTED SENTENCES ==========\n")
print(highlighted_text)

========== ORIGINAL TEXT ==========

Wal-Mart, the largest US retailer, has said its December sales are expected to be better than previously forecast because of strong post-Christmas sales."The continuing economic expansion, combined with job growth, has consumers ending this year on a high note," said Lynn Franco, director of the Conference Board's consumer research centre.The feel-good factor among US consumers rose in December for the first time since July according to new data.Consumers' confidence in the state of the US economy is at its highest for five months and they are optimistic about 2005, an influential survey says.US retailers have reported strong sales over the past 10 days after a slow start to the crucial festive season.

========== EXTRACTIVE SUMMARY ==========

walmart the largest us retailer has said its december sales are expected to be better than previously forecast because of strong postchristmas sales.the continuing economic expansion combined with job growth 

In [ ]:
# Ground-truth reference summary (if available in dataset)
# In many datasets, a human-written summary is provided for evaluation
# If not available, ROUGE evaluation cannot be performed

reference = None  


# =========================================================
# ROUGE EVALUATION SECTION
# =========================================================
# ROUGE is used to compare machine-generated summary with a reference (human summary)
# It measures overlap between words and phrases

if reference is not None:

    # Compute ROUGE scores for Extractive Summary
    rouge_ext = compute_rouge(reference, ext_summary)

    # Compute ROUGE scores for Abstractive Summary
    rouge_abs = compute_rouge(reference, abs_summary)

    # Display evaluation results
    print("\n========== ROUGE SCORES ==========\n")

    # Print Extractive summarization performance
    print("Extractive:", rouge_ext)
    print("Abstractive:", rouge_abs)

In [ ]:
# Save summaries to a text file
def save_summary(text, ext, abs_sum):
    with open("summary_output.txt", "w", encoding="utf-8") as f:
        f.write("ORIGINAL:\n" + text + "\n\n")
        f.write("EXTRACTIVE:\n" + ext + "\n\n")
        f.write("ABSTRACTIVE:\n" + abs_sum)

save_summary(text, ext_summary, abs_summary)

In [ ]:
# length control function used to show and select the length of summary (Short / Medium / Long ) based on user selection in the Streamlit app.
def length_control(option):
    if option == "short":
        # Short: max 40 words, min 15 words
        return 40, 15
    elif option == "medium":
        # Medium: max 80 words, min 30 words
        return 80, 30
    else:
        # Long: max 120 words, min 50 words
        return 120, 50


max_len, min_len = length_control("medium")
abs_summary = abstractive_summary(text, max_len, min_len)

In [ ]:
# a complete version of the code with all the improvements and features implemented, to be used in the Streamlit app and for deployment
import re
import numpy as np
import nltk
from nltk.tokenize import sent_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from transformers import pipeline
from rouge_score import rouge_scorer

nltk.download("punkt")

# Load model once
summarizer = pipeline("summarization", model="t5-small")


def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z0-9. ]', '', text)

    sentences = sent_tokenize(text)

    return text, sentences


def extractive_summary(text, num_sentences=2):

    _, sentences = preprocess_text(text)

    vectorizer = TfidfVectorizer()
    tfidf = vectorizer.fit_transform(sentences)

    scores = np.array(tfidf.sum(axis=1)).flatten()
    ranked = np.argsort(scores)[::-1]

    selected = sorted(ranked[:num_sentences])

    summary = " ".join([sentences[i] for i in selected])

    return summary, selected


def abstractive_summary(text, max_len=80, min_len=30):

    text = text[:1000]

    result = summarizer(
        text,
        max_length=max_len,
        min_length=min_len,
        do_sample=False
    )

    return result[0]["summary_text"]


def compute_rouge(ref, gen):

    scorer = rouge_scorer.RougeScorer(
        ["rouge1", "rouge2", "rougeL"],
        use_stemmer=True
    )

    return scorer.score(ref, gen)

[nltk_data] Downloading package punkt to C:\Users\Saif
[nltk_data]     Ullah\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
